In [1]:
import os
import qdrant_client


from docling.datamodel.document import DoclingDocument
from docling.chunking import HybridChunker

from llama_index.core.schema import TextNode
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.vector_stores.qdrant import QdrantVectorStore
from llama_index.core import VectorStoreIndex, StorageContext
from llama_index.core.node_parser import SentenceWindowNodeParser, get_leaf_nodes, MarkdownNodeParser
from llama_index.core.storage.docstore import SimpleDocumentStore
from llama_index.core import (
    VectorStoreIndex,
    StorageContext,
    Settings,
    Document,
)

import shutil

from qdrant_client.models import (
    VectorParams,
    Distance,
    SparseVectorParams,
)

import importlib
import utils as utils
utils = importlib.reload(utils)
from utils import iniezionetagimmagini, riassuntodocumento, metadata_extraction
import sqlite3


In [2]:
collectionname = "WAMASWINDOW"

url_embedder = os.getenv("VLLM_API_BASE_URL")
url_qdrant = os.getenv("QDRANT_URL")

client = qdrant_client.QdrantClient(url=url_qdrant)

embed_model = OpenAIEmbedding(
    api_base=url_embedder,
    model_name="BAAI/bge-m3",
    api_key="null",
)
Settings.embed_model = embed_model

vector_store = QdrantVectorStore(
    client=client,
    collection_name=collectionname,
    enable_hybrid=True,
    dense_vector_name="bge_m3",
    sparse_vector_name="bm25",
    fastembed_sparse_model="Qdrant/bm25"
)

docstore = SimpleDocumentStore()
storage_context = StorageContext.from_defaults(vector_store=vector_store, docstore=docstore)


In [3]:
if os.path.exists(f"../{collectionname}.db"):
    os.remove(f"../{collectionname}.db")

conn = sqlite3.connect(f"../{collectionname}.db", check_same_thread=False)
cursor = conn.cursor()

cursor.execute("""
    CREATE TABLE IF NOT EXISTS document_chunks (
        filename TEXT,
        chunk_index INTEGER,
        text_content TEXT,
        PRIMARY KEY (filename, chunk_index)
    )
""")
conn.commit()

In [4]:
if client.collection_exists(collectionname):
    client.delete_collection(collectionname)

client.create_collection(
    collection_name=collectionname,
    vectors_config={
        "bge_m3": VectorParams(
            size=1024,
            distance=Distance.COSINE
        )
    },
    sparse_vectors_config={
        "bm25": SparseVectorParams()
    }
)

True

In [5]:
def qdrant_embedding_window_chunking(DOC_SOURCE, DOC_SOURCE_MD):
    #caricamento del documento json
    doc = DoclingDocument.load_from_json(DOC_SOURCE)
    #metto il tag immagine
    doc = iniezionetagimmagini(doc)
    
    nodes = []
    sql_data = []
    DOC_SOURCE = DOC_SOURCE.split("/")[-1].replace(".json", ".pdf")
    
    chunker = HybridChunker()
    chunks = list(chunker.chunk(doc))
    
    for i, chunk in enumerate(chunks):
        enriched_text = chunker.contextualize(chunk=chunk)
        clean_meta = metadata_extraction(chunk)
        clean_meta["chunk_index"] = i
        
        sql_data.append((clean_meta.get("origin_filename", ""), i, enriched_text))
        
        # Aggiungi window context manualmente
        window_size = 2
        window_chunks = []
        
        # Prendi chunk precedenti
        for j in range(max(0, i - window_size), i):
            window_chunks.append(chunker.contextualize(chunk=chunks[j]))
        
        # Chunk corrente
        window_chunks.append(enriched_text)
        
        # Prendi chunk successivi
        for j in range(i + 1, min(len(chunks), i + window_size + 1)):
            window_chunks.append(chunker.contextualize(chunk=chunks[j]))
        
        window_text = "\n---\n".join(window_chunks)
        
        # Aggiungi il window ai metadata
        clean_meta["window"] = window_text
        clean_meta["original_text"] = enriched_text
        
        new_node = TextNode(
            text=enriched_text,  # Il testo principale rimane il chunk
            metadata=clean_meta
        )
        nodes.append(new_node)
    
    cursor.executemany(
        "INSERT OR REPLACE INTO document_chunks (filename, chunk_index, text_content) VALUES (?, ?, ?)", 
        sql_data
    )
    conn.commit()
    
    # INDICIZZAZIONE DEI NODI IN QDRANT
    index = VectorStoreIndex(
        nodes,
        storage_context=storage_context,
        show_progress=True
    )
    print(f"documento {DOC_SOURCE} embeddato - {len(nodes)} nodi")

In [42]:
base_folder = "../preprocessing/scratch"

for root, dirs, files in os.walk(base_folder):
    json_file = None
    md_file = None

    for file in files:
        if file.endswith(".json"):
            json_file = os.path.join(root, file)
        elif file.endswith(".md"):
            md_file = os.path.join(root, file)

    if json_file and md_file:
        qdrant_embedding_window_chunking(
            DOC_SOURCE=json_file,
            DOC_SOURCE_MD=md_file
        )


Generating embeddings:   0%|          | 0/21 [00:00<?, ?it/s]

documento PUB_Criteri_tonalizzazione_gestione_sottoscelte.pdf embeddato - 21 nodi


Token indices sequence length is longer than the specified maximum sequence length for this model (636 > 512). Running this sequence through the model will result in indexing errors


Generating embeddings:   0%|          | 0/16 [00:00<?, ?it/s]

documento UTL_Refresh_Reset_OT.pdf embeddato - 16 nodi


...

Token indices sequence length is longer than the specified maximum sequence length for this model (750 > 512). Running this sequence through the model will result in indexing errors


Generating embeddings:   0%|          | 0/26 [00:00<?, ?it/s]

documento LOG_Creazione_carico.pdf embeddato - 26 nodi


Generating embeddings:   0%|          | 0/80 [00:00<?, ?it/s]

documento UTL_WAMAS_Functional_Specifications.pdf embeddato - 80 nodi
